In [1]:
import sys; print(sys.executable) 

/home/cjen/mySageMaker/ML/customer_sentiment/.venv/bin/python


In [2]:
!mkdir -p data

import boto3

region = boto3.Session().region_name
s3 = boto3.client("s3", region_name=region)

s3.download_file("sagemaker-sample-files", "datasets/text/SST2/sst2.train", "data/train")
s3.download_file("sagemaker-sample-files", "datasets/text/SST2/sst2.test", "data/test")
print("Download complete")

Download complete


In [5]:
from sagemaker import Session, s3

session = Session()

### Upload the data

In [6]:
import boto3

region = boto3.Session().region_name
account = boto3.client("sts").get_caller_identity()["Account"]
bucket = f"sagemaker-{region}-{account}"

inputs = s3.S3Uploader.upload("data", "s3://{}/mxnet-gluon-sentiment-example/data".format(bucket))

print("S3 location " + inputs)

S3 location s3://sagemaker-us-west-2-493644444178/mxnet-gluon-sentiment-example/data


### Implement the training function - Run a SageMaker training job

In [11]:
from sagemaker.mxnet import MXNet

role = "arn:aws:iam::493644444178:role/SageMakerLocalExecutionRole"

m = MXNet(
    "sentiment.py",
    role=role,
    instance_count=1,
    instance_type="ml.m4.xlarge",
    framework_version="1.8.0",
    py_version="py37",
    distribution={"parameter_server": {"enabled": True}},
    hyperparameters={
        "batch-size": 8,
        "epochs": 2,
        "learning-rate": 0.01,
        "embedding-size": 50,
        "log-interval": 1000,
    },
)

In [12]:
m.fit(inputs)

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker:Creating training-job with name: mxnet-training-2026-03-19-18-25-35-718


2026-03-19 18:25:36 Starting - Starting the training job...
2026-03-19 18:26:07 Starting - Preparing the instances for training...
2026-03-19 18:26:35 Downloading - Downloading input data...
2026-03-19 18:27:00 Downloading - Downloading the training image...
2026-03-19 18:27:40 Training - Training image download completed. Training in progress....2026-03-19 18:27:55,397 sagemaker-training-toolkit INFO     Imported framework sagemaker_mxnet_container.training
2026-03-19 18:27:55,400 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2026-03-19 18:27:55,414 sagemaker_mxnet_container.training INFO     MXNet training environment: {'SM_HOSTS': '["algo-1"]', 'SM_NETWORK_INTERFACE_NAME': 'eth0', 'SM_HPS': '{"batch-size":8,"embedding-size":50,"epochs":2,"learning-rate":0.01,"log-interval":1000}', 'SM_USER_ENTRY_POINT': 'sentiment.py', 'SM_FRAMEWORK_PARAMS': '{"sagemaker_parameter_server_enabled":true}', 'SM_RESOURCE_CONFIG': '{"current_group_name":"homogeneousCl

In [13]:
predictor = m.deploy(initial_instance_count=1, instance_type="ml.m4.xlarge")

INFO:sagemaker:Repacking model artifact (s3://sagemaker-us-west-2-493644444178/mxnet-training-2026-03-19-18-25-35-718/output/model.tar.gz), script artifact (s3://sagemaker-us-west-2-493644444178/mxnet-training-2026-03-19-18-25-35-718/source/sourcedir.tar.gz), and dependencies ([]) into single tar.gz file located at s3://sagemaker-us-west-2-493644444178/mxnet-training-2026-03-19-19-00-22-200/model.tar.gz. This may take some time depending on model size...
INFO:sagemaker:Creating model with name: mxnet-training-2026-03-19-19-00-22-200
INFO:sagemaker:Creating endpoint-config with name mxnet-training-2026-03-19-19-00-22-200
INFO:sagemaker:Creating endpoint with name mxnet-training-2026-03-19-19-00-22-200


-----!

In [14]:
data = [
    "this movie was extremely good .",
    "the plot was very boring .",
    "this film is so slick , superficial and trend-hoppy .",
    "i just could not watch it till the end .",
    "the movie was so enthralling !",
]

response = predictor.predict(data)
print(response)

[1, 0, 0, 0, 1]


### Cleanup

In [15]:
predictor.delete_endpoint()

INFO:sagemaker:Deleting endpoint configuration with name: mxnet-training-2026-03-19-19-00-22-200
INFO:sagemaker:Deleting endpoint with name: mxnet-training-2026-03-19-19-00-22-200
